# Part IV: Industry-Level ESG and Financial Performance Analysis

**Research Questions:**
1. Which industries are "double materiality winners" (high ESG *and* high financial performance)?
2. Which industries exhibit a tension (high ESG but low returns, or low ESG but high returns)?
3. What does this imply for ESG-conscious investor allocation?
4. Recommend one industry where an investor can achieve both competitive returns and positive ESG impact.

---
**Methodology overview:**  
Industries are scored on four dimensions: mean ESG score, mean future ROA, mean future annual return, and ESG-ROA regression coefficient (from Part III). Each industry is then placed in a 2×2 materiality matrix and classified as a winner, loser, or tension case.

## 1. Setup — Load Panel from Part III

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# ── Load panel (must have been produced by part3.ipynb) ───────────────────────
panel = pd.read_csv('esg_financial_panel_2013_2023.csv')
panel['datadate'] = pd.to_datetime(panel['datadate'])
panel['sic']      = pd.to_numeric(panel['sic'], errors='coerce')

# ── Reproduce FF48 classification (identical to part3.ipynb) ─────────────────
FF48 = [
    (1,  'Agriculture',             [(100,199),(200,299),(700,799),(910,919),(2048,2048)]),
    (2,  'Food Products',           [(2000,2046),(2050,2063),(2070,2079),(2086,2086),(2090,2092),(2095,2095),(2096,2096),(2097,2099)]),
    (3,  'Candy & Soda',            [(2064,2068),(2086,2086),(2096,2096),(2097,2097)]),
    (4,  'Beer & Liquor',           [(2080,2085),(2080,2080)]),
    (5,  'Tobacco Products',        [(2100,2199)]),
    (6,  'Recreation',              [(7800,7833),(7840,7841),(7900,7999),(7993,7993),(7997,7997)]),
    (7,  'Entertainment',           [(7812,7819),(7820,7823),(7824,7829)]),
    (8,  'Printing & Publishing',   [(2700,2749),(2770,2771),(2780,2799)]),
    (9,  'Consumer Goods',          [(2047,2047),(2391,2392),(2510,2519),(2590,2599),(2840,2844),(2846,2849),(3160,3162),(3170,3172),(3190,3199),(3229,3229),(3260,3260),(3262,3263),(3269,3269),(3630,3639),(3750,3751),(3800,3800),(3860,3861),(3870,3879),(3910,3919),(3960,3969),(3991,3991),(3995,3995)]),
    (10, 'Apparel',                 [(2300,2390),(3020,3021),(3100,3111),(3130,3131),(3140,3149),(3150,3151),(3963,3965)]),
    (11, 'Healthcare',              [(8000,8099)]),
    (12, 'Medical Equipment',       [(3841,3851),(3841,3841),(3842,3842),(3843,3843),(3844,3844),(3845,3845),(3851,3851),(5047,5047),(5122,5122)]),
    (13, 'Pharmaceutical Products', [(2830,2836)]),
    (14, 'Chemicals',               [(2800,2829),(2850,2879),(2890,2899)]),
    (15, 'Rubber & Plastic',        [(3000,3030),(3041,3041),(3050,3053),(3060,3069),(3070,3079),(3080,3089),(3090,3099)]),
    (16, 'Textiles',                [(2200,2284),(2290,2295),(2297,2299),(2393,2395),(2397,2399)]),
    (17, 'Construction Materials',  [(800,899),(1500,1511),(1520,1542),(1550,1559),(1600,1699),(1711,1799),(2400,2439),(2450,2459),(2490,2499),(2660,2661),(2950,2952),(3200,3200),(3210,3211),(3240,3241),(3250,3259),(3261,3261),(3264,3264),(3270,3275),(3280,3281),(3290,3293),(3295,3299),(3420,3429),(3430,3433),(3440,3441),(3442,3442),(3446,3446),(3448,3448),(3449,3449),(3460,3469),(3490,3499),(3996,3996)]),
    (18, 'Steel Works',             [(3300,3300),(3310,3317),(3320,3325),(3330,3341),(3350,3357),(3360,3369),(3390,3399)]),
    (19, 'Fabricated Products',     [(3410,3412),(3443,3443),(3444,3444),(3460,3462),(3490,3492),(3494,3495),(3496,3499)]),
    (20, 'Machinery',               [(3510,3519),(3520,3529),(3530,3539),(3540,3549),(3550,3559),(3560,3569),(3580,3589),(3590,3599)]),
    (21, 'Electrical Equipment',    [(3600,3600),(3610,3613),(3620,3621),(3623,3629),(3640,3646),(3648,3649),(3660,3660),(3690,3692),(3699,3699)]),
    (22, 'Automobiles & Trucks',    [(2296,2296),(2396,2396),(3010,3011),(3537,3537),(3647,3647),(3694,3694),(3700,3716),(3750,3751),(3790,3792),(3799,3799)]),
    (23, 'Aircraft',                [(3720,3721),(3723,3725),(3728,3729)]),
    (24, 'Shipbuilding',            [(3730,3731),(3740,3743)]),
    (25, 'Defense',                 [(3760,3769),(3480,3489),(3812,3812)]),
    (26, 'Precious Metals',         [(1040,1049)]),
    (27, 'Non-Metallic Mining',     [(1000,1039),(1050,1099),(1400,1499)]),
    (28, 'Coal',                    [(1200,1299)]),
    (29, 'Petroleum & Natural Gas', [(1300,1399),(2900,2912),(2990,2999)]),
    (30, 'Utilities',               [(4900,4942),(4950,4952),(4953,4953),(4959,4959),(4961,4961),(4991,4991)]),
    (31, 'Communication',           [(4800,4899)]),
    (32, 'Personal Services',       [(7020,7021),(7040,7041),(7080,7081),(7200,7299),(7300,7300),(7389,7389),(7395,7395),(7500,7521),(7532,7534),(7536,7599),(7600,7641),(7690,7699),(8200,8499),(8600,8699),(8800,8899),(7510,7515)]),
    (33, 'Business Services',       [(7370,7379),(7380,7399),(7514,7515),(7519,7519),(8700,8748),(8900,8999),(4220,4229)]),
    (34, 'Computers',               [(3570,3579),(3680,3689),(3695,3695),(7373,7373)]),
    (35, 'Electronic Equipment',    [(3622,3622),(3661,3669),(3670,3679),(3810,3810),(3812,3812)]),
    (36, 'Measuring Instruments',   [(3811,3811),(3820,3829),(3830,3839)]),
    (37, 'Misc. Business',          [(3900,3999)]),
    (38, 'Transportation',          [(4000,4013),(4040,4049),(4100,4231),(4400,4499),(4600,4621),(4700,4789)]),
    (39, 'Wholesale',               [(5000,5199)]),
    (40, 'Retail',                  [(5200,5999)]),
    (41, 'Restaurants & Hotels',    [(5800,5829),(5890,5899),(7000,7011),(7041,7041)]),
    (42, 'Banking',                 [(6000,6199)]),
    (43, 'Insurance',               [(6300,6411)]),
    (44, 'Real Estate',             [(6500,6515),(6552,6553),(6798,6798)]),
    (45, 'Trading',                 [(6200,6299),(6710,6799)]),
]

sic_to_ff48 = {}
for ff_id, ff_name, ranges in FF48:
    for lo, hi in ranges:
        for s in range(lo, hi + 1):
            if s not in sic_to_ff48:
                sic_to_ff48[s] = (ff_id, ff_name)

def assign_ff48(sic):
    if pd.isna(sic): return (48, 'Other')
    return sic_to_ff48.get(int(sic), (48, 'Other'))

ff48_result      = panel['sic'].apply(assign_ff48)
panel['ff48_id'] = ff48_result.apply(lambda x: x[0]).astype(int)
panel['industry']= ff48_result.apply(lambda x: x[1])

# ── Reproduce derived variables and winsorization ─────────────────────────────
panel = panel.sort_values(['gvkey', 'fyear']).copy()

panel['leverage']          = (panel['dltt'].fillna(0) + panel['dlc'].fillna(0)) / panel['at']
panel['rd_intensity']      = panel['xrd'].fillna(0) / panel['at']
panel['capital_intensity'] = panel['ppent'].fillna(0) / panel['at']
panel['equity_ratio']      = panel['ceq'] / panel['at']
panel['profit_margin']     = np.where(panel['sale'] > 0, panel['ni'] / panel['sale'], np.nan)
panel['lag_sale']          = panel.groupby('gvkey')['sale'].shift(1)
panel['sales_growth']      = np.where(panel['lag_sale'] > 0,
                                       (panel['sale'] - panel['lag_sale']) / panel['lag_sale'], np.nan)

def winsorize(series, lower=0.01, upper=0.99):
    return series.clip(series.quantile(lower), series.quantile(upper))

for v in ['roa', 'size', 'leverage', 'rd_intensity', 'capital_intensity',
          'equity_ratio', 'profit_margin', 'sales_growth',
          'future_roa', 'future_annual_ret']:
    panel[v] = winsorize(panel[v])

panel['earnings_news'] = panel['future_roa'] - panel['roa']
panel['fyear_str']     = panel['fyear'].astype(str)
panel['ff48_id_str']   = panel['ff48_id'].astype(str)

print(f'Panel loaded: {len(panel):,} firm-year observations')
print(f'Industries:   {panel["industry"].nunique()}')
print(f'Years:        {panel["fyear"].min()} – {panel["fyear"].max()}')

## 2. Industry-Level Summary Statistics

Compute mean ESG score, future ROA, future annual return, and firm count per FF48 industry.  
Require at least 30 firm-year observations to keep industry estimates stable.

In [ ]:
# ── Aggregate to industry level ───────────────────────────────────────────────
ind_stats = (
    panel
    .groupby(['ff48_id', 'industry'])
    .agg(
        n_obs            = ('gvkey',            'count'),
        n_firms          = ('gvkey',            'nunique'),
        mean_esg         = ('esg_score',        'mean'),
        median_esg       = ('esg_score',        'median'),
        mean_future_roa  = ('future_roa',       'mean'),
        median_future_roa= ('future_roa',       'median'),
        mean_ret         = ('future_annual_ret','mean'),
        median_ret       = ('future_annual_ret','median'),
        mean_roa         = ('roa',              'mean'),
        mean_size        = ('size',             'mean'),
        mean_leverage    = ('leverage',         'mean'),
        mean_rd          = ('rd_intensity',     'mean'),
    )
    .reset_index()
)

# Drop industries with fewer than 30 obs — too noisy to interpret
ind_stats = ind_stats[ind_stats['n_obs'] >= 30].copy()
print(f'Industries with ≥30 observations: {len(ind_stats)}')

# ── Standardise dimensions for the materiality matrix ────────────────────────
# Z-score each dimension so ESG and ROA are on the same scale
for col in ['mean_esg', 'mean_future_roa', 'mean_ret']:
    mu  = ind_stats[col].mean()
    sd  = ind_stats[col].std()
    ind_stats[col + '_z'] = (ind_stats[col] - mu) / sd

# ── Assign quadrant using cross-industry medians as thresholds ────────────────
esg_med = ind_stats['mean_esg'].median()
roa_med = ind_stats['mean_future_roa'].median()
ret_med = ind_stats['mean_ret'].median()

def quadrant(row):
    hi_esg = row['mean_esg'] >= esg_med
    hi_roa = row['mean_future_roa'] >= roa_med
    if   hi_esg and hi_roa:  return 'Double winner'
    elif hi_esg and not hi_roa: return 'High ESG / Low ROA'
    elif not hi_esg and hi_roa: return 'Low ESG / High ROA'
    else:                     return 'Double laggard'

ind_stats['quadrant'] = ind_stats.apply(quadrant, axis=1)

print(f'\nMedian ESG (cross-industry): {esg_med:.4f}')
print(f'Median future ROA:           {roa_med:.4f}')
print(f'Median future return:        {ret_med:.4f}')
print(f'\nQuadrant counts:')
print(ind_stats['quadrant'].value_counts().to_string())

## 3. Full Industry Summary Table

In [ ]:
display_cols = ['industry', 'n_firms', 'n_obs',
                'mean_esg', 'mean_future_roa', 'mean_ret', 'quadrant']

table = (
    ind_stats[display_cols]
    .sort_values('mean_esg', ascending=False)
    .rename(columns={
        'industry':        'Industry',
        'n_firms':         'Firms',
        'n_obs':           'Obs',
        'mean_esg':        'Mean ESG',
        'mean_future_roa': 'Mean Future ROA',
        'mean_ret':        'Mean Future Ret',
        'quadrant':        'Quadrant',
    })
)

pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 30)
display(table.reset_index(drop=True))

## 4. Materiality Matrix — ESG vs Future ROA

Each bubble is one FF48 industry. Size = number of firms. Color = quadrant classification.  
The dashed lines mark the cross-industry medians.

In [ ]:
COLORS = {
    'Double winner':    '#1a7744',   # dark green
    'High ESG / Low ROA': '#e67e22', # orange — tension
    'Low ESG / High ROA': '#2980b9', # blue — tension
    'Double laggard':   '#c0392b',   # red
}

fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

for quad, grp in ind_stats.groupby('quadrant'):
    sizes = (grp['n_firms'] / ind_stats['n_firms'].max()) * 1800 + 80
    ax.scatter(
        grp['mean_esg'], grp['mean_future_roa'],
        s=sizes, c=COLORS[quad], alpha=0.82,
        label=quad, edgecolors='white', linewidths=0.8, zorder=3
    )

# Label each bubble
for _, row in ind_stats.iterrows():
    name = row['industry'] if len(row['industry']) <= 18 else row['industry'][:16] + '..'
    ax.annotate(
        name,
        xy=(row['mean_esg'], row['mean_future_roa']),
        xytext=(4, 4), textcoords='offset points',
        fontsize=7.2, color='#2c3e50', alpha=0.88
    )

# Median reference lines
ax.axvline(esg_med, color='#7f8c8d', lw=1.4, ls='--', alpha=0.7, label='Median ESG')
ax.axhline(roa_med, color='#7f8c8d', lw=1.4, ls=':',  alpha=0.7, label='Median ROA')

# Quadrant labels
xlim = ax.get_xlim(); ylim = ax.get_ylim()
kw = dict(fontsize=9, fontweight='bold', alpha=0.25, ha='center', va='center')
ax.text((esg_med + xlim[1])/2, (roa_med + ylim[1])/2, 'DOUBLE WINNER',   color='#1a7744', **kw)
ax.text((esg_med + xlim[0])/2, (roa_med + ylim[1])/2, 'LOW ESG\nHIGH ROA', color='#2980b9', **kw)
ax.text((esg_med + xlim[1])/2, (roa_med + ylim[0])/2, 'HIGH ESG\nLOW ROA', color='#e67e22', **kw)
ax.text((esg_med + xlim[0])/2, (roa_med + ylim[0])/2, 'DOUBLE LAGGARD',  color='#c0392b', **kw)

ax.set_xlabel('Mean ESG Score (industry average)', fontsize=11)
ax.set_ylabel('Mean Future ROA (industry average)', fontsize=11)
ax.set_title('Double Materiality Matrix: ESG Score vs Future ROA\n'
             'Bubble size = number of firms in industry | FF48 classification | 2013–2023',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.18)
plt.tight_layout()
plt.show()

## 5. ESG vs Future Return — Tension Analysis

A separate scatter comparing ESG to stock returns (rather than ROA) to identify where  
financial materiality (returns) and impact materiality (ESG) are aligned vs in tension.

In [ ]:
def quadrant_ret(row):
    hi_esg = row['mean_esg'] >= esg_med
    hi_ret = row['mean_ret'] >= ret_med
    if   hi_esg and hi_ret:      return 'High ESG / High Return'
    elif hi_esg and not hi_ret:  return 'High ESG / Low Return'
    elif not hi_esg and hi_ret:  return 'Low ESG / High Return'
    else:                        return 'Low ESG / Low Return'

ind_stats['quadrant_ret'] = ind_stats.apply(quadrant_ret, axis=1)

RET_COLORS = {
    'High ESG / High Return': '#1a7744',
    'High ESG / Low Return':  '#e67e22',
    'Low ESG / High Return':  '#2980b9',
    'Low ESG / Low Return':   '#c0392b',
}

fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

for quad, grp in ind_stats.groupby('quadrant_ret'):
    sizes = (grp['n_firms'] / ind_stats['n_firms'].max()) * 1800 + 80
    ax.scatter(
        grp['mean_esg'], grp['mean_ret'],
        s=sizes, c=RET_COLORS[quad], alpha=0.82,
        label=quad, edgecolors='white', linewidths=0.8, zorder=3
    )

for _, row in ind_stats.iterrows():
    name = row['industry'] if len(row['industry']) <= 18 else row['industry'][:16] + '..'
    ax.annotate(name, xy=(row['mean_esg'], row['mean_ret']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7.2, color='#2c3e50', alpha=0.88)

ax.axvline(esg_med, color='#7f8c8d', lw=1.4, ls='--', alpha=0.7)
ax.axhline(ret_med, color='#7f8c8d', lw=1.4, ls=':',  alpha=0.7)

ax.set_xlabel('Mean ESG Score', fontsize=11)
ax.set_ylabel('Mean Future Annual Return', fontsize=11)
ax.set_title('ESG Score vs Future Stock Returns by Industry\n'
             'Identifies where ESG and return signals are aligned vs in tension | 2013–2023',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.18)
plt.tight_layout()
plt.show()

print('\nReturn quadrant breakdown:')
print(ind_stats['quadrant_ret'].value_counts().to_string())

## 6. Industry-Level ESG → ROA Regressions

Run a separate OLS regression (ESG → future ROA, with year FEs) for each industry  
with sufficient observations. The ESG coefficient tells us whether ESG is financially  
material *within* each industry, not just in the pooled sample.

In [ ]:
MIN_OBS = 50   # minimum obs to run a stable within-industry regression

results = []
for (ff48_id, industry), grp in panel.groupby(['ff48_id', 'industry']):
    sub = grp.dropna(subset=['future_roa', 'esg_score', 'fyear_str'])
    if len(sub) < MIN_OBS:
        continue
    # Only add year FE if we have enough years to identify
    n_years = sub['fyear_str'].nunique()
    formula = (
        'future_roa ~ esg_score + C(fyear_str)'
        if n_years >= 3
        else 'future_roa ~ esg_score'
    )
    try:
        m = smf.ols(formula, data=sub).fit(cov_type='HC3')
        results.append({
            'ff48_id':  ff48_id,
            'industry': industry,
            'n_obs':    len(sub),
            'esg_coef': m.params['esg_score'],
            'esg_se':   m.bse['esg_score'],
            'esg_t':    m.tvalues['esg_score'],
            'esg_p':    m.pvalues['esg_score'],
            'r2':       m.rsquared,
        })
    except Exception:
        pass

reg_results = pd.DataFrame(results)
reg_results['sig'] = reg_results['esg_p'].apply(
    lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
)

# Merge back into ind_stats
ind_stats = ind_stats.merge(
    reg_results[['ff48_id', 'esg_coef', 'esg_p', 'sig']],
    on='ff48_id', how='left'
)

print(f'Within-industry regressions run: {len(reg_results)}')
print(f'\nTop 10 industries by ESG→ROA coefficient:')
top10 = reg_results.nlargest(10, 'esg_coef')[['industry','n_obs','esg_coef','esg_p','sig','r2']]
print(top10.to_string(index=False))

## 7. ESG Coefficient Bar Chart — Within-Industry Financial Materiality

In [ ]:
plot_df = reg_results.dropna(subset=['esg_coef']).sort_values('esg_coef', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(plot_df) * 0.32)))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

colors = ['#1a7744' if c > 0 else '#c0392b' for c in plot_df['esg_coef']]
bars = ax.barh(plot_df['industry'], plot_df['esg_coef'], color=colors, alpha=0.85, height=0.7)

# Significance stars
for i, (_, row) in enumerate(plot_df.iterrows()):
    if row['sig']:
        xpos = row['esg_coef'] + (0.003 if row['esg_coef'] >= 0 else -0.003)
        ha   = 'left' if row['esg_coef'] >= 0 else 'right'
        ax.text(xpos, i, row['sig'], va='center', ha=ha, fontsize=8, color='#2c3e50')

ax.axvline(0, color='#2c3e50', lw=1)
ax.set_xlabel('ESG → Future ROA coefficient (within-industry OLS, year FEs, HC3 SE)', fontsize=10)
ax.set_title('Within-Industry ESG Financial Materiality\n'
             'Positive = higher ESG predicts higher future ROA | * p<0.10  ** p<0.05  *** p<0.01',
             fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Double Materiality Winners and Tension Cases — Summary Table

In [ ]:
summary_cols = ['industry', 'n_firms', 'mean_esg', 'mean_future_roa',
                'mean_ret', 'esg_coef', 'sig', 'quadrant']

summary = ind_stats[summary_cols].copy().rename(columns={
    'industry':        'Industry',
    'n_firms':         'Firms',
    'mean_esg':        'ESG',
    'mean_future_roa': 'Future ROA',
    'mean_ret':        'Future Ret',
    'esg_coef':        'ESG→ROA coef',
    'sig':             'Sig',
    'quadrant':        'Quadrant',
})

print('=== DOUBLE MATERIALITY WINNERS (High ESG + High ROA) ===')
winners = summary[summary['Quadrant'] == 'Double winner'].sort_values('ESG', ascending=False)
display(winners.reset_index(drop=True))

print('\n=== TENSION CASE: High ESG / Low ROA ===')
t1 = summary[summary['Quadrant'] == 'High ESG / Low ROA'].sort_values('ESG', ascending=False)
display(t1.reset_index(drop=True))

print('\n=== TENSION CASE: Low ESG / High ROA ===')
t2 = summary[summary['Quadrant'] == 'Low ESG / High ROA'].sort_values('Future ROA', ascending=False)
display(t2.reset_index(drop=True))

print('\n=== DOUBLE LAGGARDS (Low ESG + Low ROA) ===')
laggers = summary[summary['Quadrant'] == 'Double laggard'].sort_values('ESG')
display(laggers.reset_index(drop=True))

## 9. Recommended Industry — Descriptive Statistics

Select the recommended industry as the double-winner with the highest within-industry  
ESG→ROA coefficient that is also statistically significant. Print full firm-level  
descriptive statistics for that industry.

In [ ]:
# ── Identify the recommended industry programmatically ───────────────────────
# Criteria (in order): (1) double winner quadrant, (2) significant ESG→ROA
# coefficient, (3) highest ESG coef among qualifying industries.

candidates = ind_stats[
    (ind_stats['quadrant'] == 'Double winner') &
    (ind_stats['esg_p']    <= 0.10)            # at least marginal significance
].copy()

if len(candidates) == 0:
    # Fall back to all double winners if none are significant
    candidates = ind_stats[ind_stats['quadrant'] == 'Double winner'].copy()
    print('Note: No double winners with significant ESG coef — using all double winners.')

# Among candidates, pick the one with the highest ESG→ROA coefficient
best = candidates.sort_values('esg_coef', ascending=False).iloc[0]
REC_INDUSTRY = best['industry']
REC_FF48     = best['ff48_id']

print(f'Recommended industry: {REC_INDUSTRY} (FF48 id = {int(REC_FF48)})')
print(f'  Mean ESG score:     {best["mean_esg"]:.4f}')
print(f'  Mean future ROA:    {best["mean_future_roa"]:.4f}')
print(f'  Mean future return: {best["mean_ret"]:.4f}')
print(f'  ESG→ROA coef:       {best["esg_coef"]:.4f}  {best["sig"]}')
print(f'  N firms:            {int(best["n_firms"])}')

In [ ]:
# ── Firm-level descriptive statistics for the recommended industry ────────────
rec_panel = panel[panel['ff48_id'] == REC_FF48].copy()

desc_vars = [
    'esg_score', 'future_roa', 'future_annual_ret', 'roa',
    'size', 'leverage', 'rd_intensity', 'capital_intensity',
    'sales_growth', 'profit_margin'
]

desc = rec_panel[desc_vars].describe(percentiles=[0.25, 0.5, 0.75]).T
desc.index.name = 'Variable'
desc.columns    = ['N', 'Mean', 'Std', 'Min', 'P25', 'Median', 'P75', 'Max']

print(f'=== Descriptive Statistics: {REC_INDUSTRY} ===')
print(f'    {len(rec_panel):,} firm-year observations  |  '
      f'{rec_panel["gvkey"].nunique():,} unique firms  |  '
      f'{rec_panel["fyear"].min()}–{rec_panel["fyear"].max()}')
print()
pd.set_option('display.float_format', '{:.4f}'.format)
display(desc)

In [ ]:
# ── Time-series plot: ESG score and ROA trends in the recommended industry ────
ts = (
    rec_panel
    .groupby('fyear')
    .agg(mean_esg=('esg_score','mean'), mean_roa=('roa','mean'),
         mean_future_roa=('future_roa','mean'))
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#f8f9fa')
ax1.set_facecolor('#f8f9fa')

ax2 = ax1.twinx()

l1, = ax1.plot(ts['fyear'], ts['mean_esg'],       '-o', color='#1a7744', lw=2.2,
               ms=6, label='Mean ESG score (left)')
l2, = ax2.plot(ts['fyear'], ts['mean_roa'],        '-s', color='#2980b9', lw=2,
               ms=6, label='Mean ROA (right)')
l3, = ax2.plot(ts['fyear'], ts['mean_future_roa'], '--^', color='#8e44ad', lw=1.8,
               ms=5, label='Mean future ROA (right)')

ax1.set_xlabel('Fiscal Year', fontsize=11)
ax1.set_ylabel('Mean ESG Score', fontsize=11, color='#1a7744')
ax2.set_ylabel('Mean ROA', fontsize=11, color='#2980b9')
ax1.set_title(f'Time-Series: ESG and Profitability Trends — {REC_INDUSTRY}',
              fontsize=12, fontweight='bold')
ax1.legend(handles=[l1, l2, l3], fontsize=9, loc='upper left')
ax1.grid(alpha=0.18)
plt.tight_layout()
plt.show()

In [ ]:
# ── ESG score distribution in recommended industry vs all others ──────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#f8f9fa')

# ESG distribution comparison
ax = axes[0]
ax.set_facecolor('#f8f9fa')
other = panel[panel['ff48_id'] != REC_FF48]['esg_score'].dropna()
rec   = rec_panel['esg_score'].dropna()
ax.hist(other, bins=40, alpha=0.5, color='#7f8c8d', label='All other industries', density=True)
ax.hist(rec,   bins=40, alpha=0.75, color='#1a7744', label=REC_INDUSTRY, density=True)
ax.axvline(other.mean(), color='#7f8c8d', lw=1.5, ls='--')
ax.axvline(rec.mean(),   color='#1a7744', lw=1.5, ls='--')
ax.set_xlabel('ESG Score'); ax.set_ylabel('Density')
ax.set_title('ESG Score Distribution', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.18)

# ROA distribution comparison
ax = axes[1]
ax.set_facecolor('#f8f9fa')
other_roa = panel[panel['ff48_id'] != REC_FF48]['future_roa'].dropna()
rec_roa   = rec_panel['future_roa'].dropna()
ax.hist(other_roa, bins=40, alpha=0.5, color='#7f8c8d', label='All other industries', density=True)
ax.hist(rec_roa,   bins=40, alpha=0.75, color='#2980b9', label=REC_INDUSTRY, density=True)
ax.axvline(other_roa.mean(), color='#7f8c8d', lw=1.5, ls='--')
ax.axvline(rec_roa.mean(),   color='#2980b9', lw=1.5, ls='--')
ax.set_xlabel('Future ROA'); ax.set_ylabel('Density')
ax.set_title('Future ROA Distribution', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.18)

fig.suptitle(f'Recommended Industry vs Rest of Sample: {REC_INDUSTRY}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Investment Implications — Discussion Framework

### Double materiality winners
These industries satisfy both financial materiality (ESG predicts ROA) and impact materiality  
(high absolute ESG scores indicate positive real-world outcomes). ESG-conscious investors face  
no trade-off here: allocating to these industries is consistent with both return-seeking and  
impact objectives. The positive within-industry ESG→ROA coefficient further implies that  
*active* ESG selection within the industry — picking the higher-ESG firms — is rewarded.

### High ESG / Low ROA tension ("impact premium industries")
These are industries where doing good and doing well pull in opposite directions.  
Possible explanations include: (a) ESG investment is costly and reduces margins (the  
"doing good is expensive" view); (b) the market has already overpriced ESG quality in  
these sectors, depressing future returns; (c) ESG scores reflect genuine impact (e.g.  
environmental compliance) that doesn't translate to accounting profitability.  
ESG-conscious investors in these industries should expect to pay an "impact premium" —  
they accept lower financial returns in exchange for higher real-world impact.

### Low ESG / High ROA tension ("sin stocks" pattern)
These industries earn high accounting returns despite (or because of) low ESG scores.  
This is consistent with the literature on sin stocks (Hong & Kacperski, 2009): neglected  
assets earn higher returns because constrained investors (those with ESG screens) cannot  
hold them, depressing prices and inflating expected returns. ESG-screened portfolios  
exclude these industries at a financial cost — a cost that should be acknowledged explicitly.

### Recommended industry — selection rationale
The recommended industry was selected on three criteria simultaneously:  
(1) above-median ESG score (impact materiality),  
(2) above-median future ROA (financial materiality — accounting),  
(3) statistically significant positive within-industry ESG→ROA coefficient (ESG drives  
performance, not just correlates with it).  

An ESG-conscious investor allocating to this industry can construct a portfolio that  
screens for higher-ESG firms within the industry and expect, based on the historical  
evidence, to earn above-average returns — not despite the ESG screen, but partly  
because of it.
